In [6]:
import json
import numpy as np
from sklearn.cluster import KMeans

In [3]:
# Store paths for all inferences

INFERENCE_PATHS = {}

INFERENCE_PATHS['NOSCHEMA'] = {
    'VALID': './outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid/prediction_raw.json',
    'TEST': './outputs/eval_ehrsql_mimic3_t5_base__mimic3_test/prediction_raw.json'
}
INFERENCE_PATHS['WITHSCHEMA'] = {
    'VALID': './outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction_raw.json',
    'TEST': './outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction_raw.json'    
}

print(INFERENCE_PATHS)




{'NOSCHEMA': {'VALID': './outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid/prediction_raw.json', 'TEST': './outputs/eval_ehrsql_mimic3_t5_base__mimic3_test/prediction_raw.json'}, 'WITHSCHEMA': {'VALID': './outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction_raw.json', 'TEST': './outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction_raw.json'}}


In [19]:
# Given a set of entropies, return thresholds based on k-means and the 67th percentile

def get_thresholds(entropy):
    
    kmeans = KMeans(n_clusters=2, random_state=0, n_init='auto').fit(np.expand_dims(entropy, axis=1))
    
    zero_low = min(np.array(entropy)[kmeans.labels_==0])
    zero_high = max(np.array(entropy)[kmeans.labels_==0])
    one_low = min(np.array(entropy)[kmeans.labels_==1])
    one_high = max(np.array(entropy)[kmeans.labels_==1])
    
    if one_high > zero_high:
        k_means_th = (zero_high + one_low)/2
    else:
        k_means_th = (zero_low + one_high)/2

    
    percent_th = np.percentile(entropy, q=[67.0])[0]

    return k_means_th, percent_th



In [20]:
# For each inference path, compute thresholds

THRESHOLDS = {}

def load_file(path):
    num_workers = -1
    with open(path, 'r') as f:
        data = json.load(f)
    print(f'[result] {len(data)} lines loaded from file ' + path)
    
    entropy = []
    for idx_, line in data.items():
        entropy.append(max(line['sequence_entropy']))
        #impossible.append(line['is_impossible'])
    return entropy


for key, value in INFERENCE_PATHS.items():
    THRESHOLDS[key] = {}
    for k, v in value.items():
        entropy = load_file(v)        
        THRESHOLDS[key][k] = get_thresholds(entropy)


print(THRESHOLDS)

[result] 1122 lines loaded from file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid/prediction_raw.json
[result] 1786 lines loaded from file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_test/prediction_raw.json
[result] 1122 lines loaded from file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction_raw.json
[result] 1786 lines loaded from file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction_raw.json
{'NOSCHEMA': {'VALID': (0.7869995534420013, 0.09535356968641318), 'TEST': (0.8691008388996124, 0.053593070991337304)}, 'WITHSCHEMA': {'VALID': (0.8798104524612427, 0.21779629901051548), 'TEST': (0.7502410709857941, 0.15381738692522048)}}


# Model Evaluations

## Without Schema - Validation Data 

In [26]:
# NO THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid --input_file prediction_raw.json --output_file prediction.nothreshold.json --threshold -1
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid/prediction.nothreshold.json


/Users/satnamgandhi/projects/gatech/CSE6250/project/EHRSQL/T5/abstain_with_entropy.py:18: UserWarning: Threshold value is not set! All predictions are sent to the database.
  warnings.warn("Threshold value is not set! All predictions are sent to the database.")
{
  "precision_ans": 67.74,
  "recall_ans": 100.0,
  "f1_ans": 80.77,
  "precision_exec": 65.51,
  "recall_exec": 96.71,
  "f1_exec": 78.11
}


In [27]:
# WITH KMEANS CLUSTERING

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid --input_file prediction_raw.json --output_file prediction.clustering.json --threshold 0.7869995534420013
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid/prediction.clustering.json


{
  "precision_ans": 81.47,
  "recall_ans": 97.76,
  "f1_ans": 88.88,
  "precision_exec": 80.15,
  "recall_exec": 96.18,
  "f1_exec": 87.44
}


In [28]:
# WITH PERCENTILE BASED THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid --input_file prediction_raw.json --output_file prediction.percentile.json --threshold 0.09535356968641318
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_valid/prediction.percentile.json


{
  "precision_ans": 95.21,
  "recall_ans": 94.21,
  "f1_ans": 94.71,
  "precision_exec": 94.41,
  "recall_exec": 93.42,
  "f1_exec": 93.92
}


## Without Schema - Test Data 

In [33]:
# NO THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base__mimic3_test --input_file prediction_raw.json --output_file prediction.nothreshold.json --threshold -1
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_test/prediction.nothreshold.json


/Users/satnamgandhi/projects/gatech/CSE6250/project/EHRSQL/T5/abstain_with_entropy.py:18: UserWarning: Threshold value is not set! All predictions are sent to the database.
  warnings.warn("Threshold value is not set! All predictions are sent to the database.")
{
  "precision_ans": 67.08,
  "recall_ans": 100.0,
  "f1_ans": 80.29,
  "precision_exec": 65.17,
  "recall_exec": 97.16,
  "f1_exec": 78.02
}


In [34]:
# WITH KMEANS CLUSTERING

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base__mimic3_test --input_file prediction_raw.json --output_file prediction.clustering.json --threshold 0.8691008388996124
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_test/prediction.clustering.json


{
  "precision_ans": 76.67,
  "recall_ans": 97.91,
  "f1_ans": 86.0,
  "precision_exec": 75.23,
  "recall_exec": 96.08,
  "f1_exec": 84.38
}


In [35]:
# WITH PERCENTILE BASED THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base__mimic3_test --input_file prediction_raw.json --output_file prediction.percentile.json --threshold 0.053593070991337304
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base__mimic3_test/prediction.percentile.json


{
  "precision_ans": 92.31,
  "recall_ans": 92.15,
  "f1_ans": 92.23,
  "precision_exec": 91.56,
  "recall_exec": 91.4,
  "f1_exec": 91.48
}


## With Schema - Validation Data 

In [37]:
# NO THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid --input_file prediction_raw.json --output_file prediction.nothreshold.json --threshold -1
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction.nothreshold.json


/Users/satnamgandhi/projects/gatech/CSE6250/project/EHRSQL/T5/abstain_with_entropy.py:18: UserWarning: Threshold value is not set! All predictions are sent to the database.
  warnings.warn("Threshold value is not set! All predictions are sent to the database.")
{
  "precision_ans": 67.74,
  "recall_ans": 100.0,
  "f1_ans": 80.77,
  "precision_exec": 65.51,
  "recall_exec": 96.71,
  "f1_exec": 78.11
}


In [38]:
# WITH KMEANS CLUSTERING

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid --input_file prediction_raw.json --output_file prediction.clustering.json --threshold 0.8798104524612427
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction.clustering.json


{
  "precision_ans": 79.45,
  "recall_ans": 98.16,
  "f1_ans": 87.82,
  "precision_exec": 77.53,
  "recall_exec": 95.79,
  "f1_exec": 85.7
}


In [41]:
# WITH PERCENTILE BASED THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid --input_file prediction_raw.json --output_file prediction.percentile.json --threshold 0.21779629901051548
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/valid.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_valid/prediction.percentile.json


{
  "precision_ans": 93.62,
  "recall_ans": 92.63,
  "f1_ans": 93.12,
  "precision_exec": 92.29,
  "recall_exec": 91.32,
  "f1_exec": 91.8
}


## With Schema - Test Data 

In [42]:
# NO THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test --input_file prediction_raw.json --output_file prediction.nothreshold.json --threshold -1
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction.nothreshold.json


/Users/satnamgandhi/projects/gatech/CSE6250/project/EHRSQL/T5/abstain_with_entropy.py:18: UserWarning: Threshold value is not set! All predictions are sent to the database.
  warnings.warn("Threshold value is not set! All predictions are sent to the database.")
{
  "precision_ans": 67.08,
  "recall_ans": 100.0,
  "f1_ans": 80.29,
  "precision_exec": 64.73,
  "recall_exec": 96.49,
  "f1_exec": 77.48
}


In [43]:
# WITH KMEANS CLUSTERING

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test --input_file prediction_raw.json --output_file prediction.clustering.json --threshold 0.7502410709857941
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction.clustering.json


{
  "precision_ans": 76.2,
  "recall_ans": 96.99,
  "f1_ans": 85.35,
  "precision_exec": 74.75,
  "recall_exec": 95.16,
  "f1_exec": 83.73
}


In [45]:
# WITH PERCENTILE BASED THRESHOLD

!python T5/abstain_with_entropy.py --inference_result_path outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test --input_file prediction_raw.json --output_file prediction.percentile.json --threshold 0.15381738692522048
!python evaluate.py --db_path ./dataset/ehrsql/mimic_iii/mimic_iii.sqlite --data_file dataset/ehrsql/mimic_iii/test.json --pred_file ./outputs/eval_ehrsql_mimic3_t5_base_schema__mimic3_test/prediction.percentile.json


{
  "precision_ans": 89.05,
  "recall_ans": 88.9,
  "f1_ans": 88.97,
  "precision_exec": 88.13,
  "recall_exec": 87.98,
  "f1_exec": 88.05
}


## END OF EVALUATION 